# Week 3: Multilayer Perceptrons and Backpropagation

**Lecture 5:** Multilayer perceptrons and the forward pass  
**Lecture 6:** Backpropagation mathematics

# Lecture 5: Multilayer Perceptrons and the Forward Pass

A multilayer perceptron (MLP) composes affine transformations with nonlinear activation functions. For layer $\ell$,

$$z^{(\ell)} = a^{(\ell-1)}W^{(\ell)} + b^{(\ell)},$$

$$a^{(\ell)} = \sigma\!\left(z^{(\ell)}\right).$$

The input is $a^{(0)}=X$. A **forward pass** evaluates these equations from the first hidden layer through the output layer. The matrices $W^{(\ell)}$ and vectors $b^{(\ell)}$ are the parameters the network will learn.

In [3]:
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# Two examples, two input features each.
X = np.array([
    [0.2, 0.8],
    [0.9, 0.1],
])

# Parameters for a 2 -> 3 -> 1 MLP.
W_1 = np.array([
    [0.4, -0.3, 0.2],
    [0.1, 0.5, -0.4],
])
b_1 = np.array([0.1, -0.2, 0.0])
W_2 = np.array([
    [0.7],
    [-0.6],
    [0.3],
])
b_2 = np.array([0.05])

z_1 = X @ W_1 + b_1
a_1 = sigmoid(z_1)
z_2 = a_1 @ W_2 + b_2
y_hat = sigmoid(z_2)

print("hidden pre-activations z_1:")
print(np.round(z_1, 3))
print("hidden activations a_1:")
print(np.round(a_1, 3))
print("network output y_hat:")
print(np.round(y_hat, 3))

hidden pre-activations z_1:
[[ 0.26  0.14 -0.28]
 [ 0.47 -0.42  0.14]]
hidden activations a_1:
[[0.565 0.535 0.43 ]
 [0.615 0.397 0.535]]
network output y_hat:
[[0.563]
 [0.599]]


The same calculation can be organized into a reusable class. This week the class only performs the forward pass; Week 4 will add gradient computation and parameter updates.

In [4]:
class MultilayerPerceptron:
    def __init__(self, layer_sizes, seed=1):
        rng = np.random.default_rng(seed)
        self.weights = []
        self.biases = []

        # Each adjacent pair of widths defines one affine layer.
        for input_size, output_size in zip(layer_sizes[:-1], layer_sizes[1:]):
            scale = np.sqrt(2 / (input_size + output_size))
            self.weights.append(
                rng.normal(0, scale, size=(input_size, output_size))
            )
            self.biases.append(np.zeros(output_size))

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-z))

    def forward(self, X):
        # Keep the input as activation a^(0) and cache every intermediate value.
        activations = [np.atleast_2d(X)]
        pre_activations = []

        for W, b in zip(self.weights, self.biases):
            # Matrix multiplication processes every example in the batch at once.
            z = activations[-1] @ W + b
            a = self.sigmoid(z)
            pre_activations.append(z)
            activations.append(a)

        return activations, pre_activations

    def predict_proba(self, X):
        activations, _ = self.forward(X)
        return activations[-1]

In [5]:
# The list describes input width, hidden width, and output width.
model = MultilayerPerceptron([2, 3, 1], seed=1)
activations, pre_activations = model.forward(X)

for layer, (z, a) in enumerate(
    zip(pre_activations, activations[1:]), start=1
):
    print(f"layer {layer}: z shape={z.shape}, a shape={a.shape}")

print("predicted probabilities:")
print(np.round(model.predict_proba(X), 3))

layer 1: z shape=(2, 3), a shape=(2, 3)
layer 2: z shape=(2, 1), a shape=(2, 1)
predicted probabilities:
[[0.568]
 [0.55 ]]


# Lecture 6: Backpropagation Mathematics

Backpropagation applies the chain rule from the output layer back toward the input. If $L$ is the loss, define the layer error

$$\delta^{(\ell)}=\frac{\partial L}{\partial z^{(\ell)}}.$$

For a sigmoid output with squared-error loss,

$$\delta^{(L)}=(a^{(L)}-y)\odot\sigma'\!\left(z^{(L)}\right).$$

For each preceding layer,

$$\delta^{(\ell)}=\left(\delta^{(\ell+1)}(W^{(\ell+1)})^T\right)\odot\sigma'\!\left(z^{(\ell)}\right).$$

The parameter gradients are

$$\frac{\partial L}{\partial W^{(\ell)}}=(a^{(\ell-1)})^T\delta^{(\ell)},\qquad
\frac{\partial L}{\partial b^{(\ell)}}=\sum_i\delta_i^{(\ell)}.$$

The full written slides are posted in the web class. For another written treatment, see Michael Nielsen's [*Neural Networks and Deep Learning*, Chapter 2](http://neuralnetworksanddeeplearning.com/chap2.html). Week 4 turns these equations into code.

## Code Takeaways

- A multilayer perceptron stores one weight matrix and bias vector for each pair of adjacent layers.
- The forward pass alternates affine transformations $z=aW+b$ with nonlinear activations.
- Batch dimension is preserved throughout the network; only the feature width changes from layer to layer.
- Saving both pre-activations $z$ and activations $a$ during the forward pass provides the intermediate values needed by backpropagation.
- Without nonlinear activations, stacking affine layers is equivalent to one affine transformation.
- The reusable class separates architecture construction, forward computation, and probability prediction before Week 4 adds parameter updates.